# Characterize nodules at their ground-truth locations with MedGemma

Runs `5b_characterize_ground_truth_nodules.py`: for every pylidc consensus nodule already recorded in `ground_truth_annotations.json` (patients 1-20), crops directly around that nodule's own real centroid/diameter (no detector involved) and asks MedGemma 1.5 to rate it on the same 9 pylidc attributes as `5_characterize_nodules.py` - then compares straight against the mean of that nodule's own radiologist annotations. No MONAI/simpleitk needed, since there's no detector step.

Before running anything:
1. **Runtime > Change runtime type > GPU** (T4 is fine).
2. Upload `lidc_idri_p1-20.zip` to your Google Drive (same zip used for the counting notebook - patients 1-20 + `annotations.csv`, ~1.1GB). Skip this if you already have it there from before.
3. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) holding a Hugging Face access token that has accepted the MedGemma license at https://huggingface.co/google/medgemma-1.5-4b-it.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/freya-gul/rail.git
%cd rail
!git checkout medgemma-characterization-variants

Cloning into 'rail'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 104 (delta 35), reused 82 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 74.22 MiB | 53.67 MiB/s, done.
Resolving deltas: 100% (35/35), done.
Updating files: 100% (51/51), done.
Encountered 2 files that should have been pointers, but weren't:
	image_download/monai_bundles/lung_nodule_ct_detection/models/model.pt
	image_download/monai_bundles/lung_nodule_ct_detection/models/model.ts
/content/rail
M	image_download/monai_bundles/lung_nodule_ct_detection/models/model.pt
M	image_download/monai_bundles/lung_nodule_ct_detection/models/model.ts
Branch 'medgemma-characterization-variants' set up to track remote branch 'medgemma-characterization-variants' from 'origin'.
Switched to a new branch 'medgemma-characterization-variants'


Point this at wherever you uploaded `lidc_idri_p1-20.zip` in Drive. It unzips into `datasets/LDIC-IDRI-subset/` inside the cloned repo — that's the path this script's `DICOM_ROOT` already expects:

In [3]:
ZIP_PATH = "/content/drive/MyDrive/lidc_idri_p1-20.zip"  # <-- update to your actual upload path
DATA_DIR = "datasets/LDIC-IDRI-subset"  # relative to the repo root (we've already %cd'd into rail)

import pathlib
assert pathlib.Path(ZIP_PATH).exists(), f"{ZIP_PATH} not found — check the path/upload"

In [4]:
!mkdir -p {DATA_DIR}
!unzip -q {ZIP_PATH} -d {DATA_DIR}
!ls {DATA_DIR}

annotations.csv  lidc_idri


In [5]:
# No monai/simpleitk needed - this script never touches the MONAI detector, only
# crops directly out of the raw DICOM series at the ground-truth nodule locations.
!pip install -q pydicom "transformers>=5.12.1" "huggingface_hub>=1.21.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.7 MB/s eta 0:00:00


In [6]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

Sanity check: GPU visible to torch (the script already defaults to `cuda` > `mps` > `cpu`):

In [7]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA available: True
NVIDIA A100-SXM4-40GB


## Run characterization

Prints live progress per nodule (predicted vs. ground-truth attribute values aren't shown inline, but pass/fail on JSON validation is) plus a running ETA. Resumable at the individual-nodule level — a killed run picks back up partway through a patient rather than redoing it.

In [8]:
!python image_download/5b_characterize_ground_truth_nodules.py --start 1 --end 20

Loading MedGemma 1.5 on cuda... (64 nodule(s) to characterize)
config.json: 100% 2.55k/2.55k [00:00<00:00, 7.85MB/s]
model.safetensors.index.json: 100% 90.6k/90.6k [00:00<00:00, 148MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.96G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/8.60G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  68% 5.84G/8.60G [00:18<00:12, 225MB/s,  163MB/s  ]
Reconstructing (incomplete total...):  69% 5.92G/8.60G [00:18<00:11, 238MB/s,  155MB/s  ]
Reconstructing (incomplete total...):  98% 8.40G/8.60G [00:23<00:00, 370MB/s,  184MB/s  ]

Fetching 2 files: 100% 2/2 [00:23<00:00, 11.75s/it]
Download complete: 100% 7.79G/7.79G [00:23<00:00, 84.8MB/s,  146MB/s  ]
Download complete: 100% 7.79G/7.79G [00:23<00:00, 330MB/s,  146MB/s  ] 
Reconstruction complete: 100% 8.60G/8.60G [00:23<00:00, 365MB/s,  226MB/s

## Peek at partial results anytime

Run this whenever you want — while the run above is still going, if it got interrupted, or once it's fully done. It reads whatever `nodule_characteristics_gt/<patient>.json` files already exist on disk and computes the same MAE/bias summary the final cell below does, over however many nodules have actually been characterized so far. No need to wait for all `--end` patients to finish, and safe to re-run repeatedly as more come in.

In [9]:
import json
import sys
from pathlib import Path

sys.path.insert(0, "image_download")
from lidc_attributes import LIDC_ATTRIBUTES

OUTPUT_DIR = Path("image_download/nodule_characteristics_gt")
patient_files = sorted(OUTPUT_DIR.glob("LIDC-IDRI-*.json"))
rows = [row for f in patient_files for row in json.loads(f.read_text())]

print(f"{len(rows)} nodule(s) characterized so far across {len(patient_files)} patient(s)\n")
print(f"{'attribute':<18}{'MAE':>8}{'bias':>8}{'n':>6}   scale")
for a in LIDC_ATTRIBUTES:
    errs = [r[f"{a}_err"] for r in rows if r.get(f"{a}_err") is not None]
    mae = sum(abs(e) for e in errs) / len(errs) if errs else None
    bias = sum(errs) / len(errs) if errs else None
    lo, hi = min(LIDC_ATTRIBUTES[a]["labels"]), max(LIDC_ATTRIBUTES[a]["labels"])
    mae_str = f"{mae:.2f}" if mae is not None else "n/a"
    bias_str = f"{bias:+.2f}" if bias is not None else "n/a"
    print(f"{a:<18}{mae_str:>8}{bias_str:>8}{len(errs):>6}   {lo}-{hi}")

64 nodule(s) characterized so far across 20 patient(s)

attribute              MAE    bias     n   scale
subtlety              1.08   -0.30    62   1-5
internalStructure     0.05   +0.05    62   1-4
calcification         0.40   +0.17    62   1-6
sphericity            0.80   -0.18    62   1-5
margin                0.82   -0.01    62   1-5
lobulation            0.95   +0.28    62   1-5
spiculation           0.64   -0.51    62   1-5
texture               0.58   +0.27    62   1-5
malignancy            0.96   -0.01    62   1-5


## Results

- `image_download/nodule_characteristics_gt/<patient>.json` — per-nodule predicted attributes, ground-truth mean, error, raw MedGemma response, and JSON-validation problems (if any).
- `image_download/characterize_ground_truth_comparison.csv` — the same data flattened across all patients, one row per nodule.
- `image_download/characterize_ground_truth_summary.json` — MAE and signed bias per attribute, aggregated across every nodule.

Quick look at the summary table:

In [10]:
import json
summary = json.load(open("image_download/characterize_ground_truth_summary.json"))
for attr, stats in summary.items():
    print(f"{attr:<18} MAE={stats['mae']:.2f}  bias={stats['bias']:+.2f}  n={stats['n']}")

subtlety           MAE=1.08  bias=-0.30  n=62
internalStructure  MAE=0.05  bias=+0.05  n=62
calcification      MAE=0.40  bias=+0.17  n=62
sphericity         MAE=0.80  bias=-0.18  n=62
margin             MAE=0.82  bias=-0.01  n=62
lobulation         MAE=0.95  bias=+0.28  n=62
spiculation        MAE=0.64  bias=-0.51  n=62
texture            MAE=0.58  bias=+0.27  n=62
malignancy         MAE=0.96  bias=-0.01  n=62


## Try prompt variants: anchored guidance / few-shot examples

Runs `5c_characterize_variants.py`, an A/B sibling of the script above. The first nodule characterized (`LIDC-IDRI-0001` #0) showed a near-worst-case spiculation miss (predicted 1 "No Spiculation" vs. ground truth 4.25 "Marked Spiculation") that matches a bias `bias_correction.json` already measured on the detector-based pipeline (subtlety -0.53, margin -0.55, spiculation -0.44). Two cheap levers to test before reaching for LoRA fine-tuning:

- `--anchored` — adds targeted anti-underrating guidance to the subtlety/margin/spiculation attribute descriptions. Free (a few extra sentences, no extra images).
- `--fewshot` — prepends two fixed calibration examples (real images + their consensus-rounded ground truth) as prior conversation turns: one unambiguous low-spiculation nodule and one unambiguous high-spiculation nodule, both 4/4-reader consensus. Costs roughly 2x the vision tokens/time per nodule.

Combinable, and each flag combination writes to its own directory (`nodule_characteristics_gt_anchored/`, `..._fewshot/`, `..._anchored_fewshot/`) so nothing clobbers the zero-shot baseline above. Try a small `--end` first (a handful of patients) before committing to the full 20 — few-shot in particular roughly doubles per-nodule runtime.

In [11]:
!python image_download/5c_characterize_variants.py --anchored --start 1 --end 5

Variant: anchored=True fewshot=False -> /content/rail/image_download/nodule_characteristics_gt_anchored
Loading MedGemma 1.5 on cuda... (10 nodule(s) to characterize)
Loading weights: 100% 883/883 [00:00<00:00, 5221.32it/s]
[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'return_dict_in_generate'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Traceback (most recent call last):
  File "/content/rail/image_download/5c_characte

In [12]:
!python image_download/5c_characterize_variants.py --fewshot --start 1 --end 5

Variant: anchored=False fewshot=True -> /content/rail/image_download/nodule_characteristics_gt_fewshot
Loading MedGemma 1.5 on cuda... (9 nodule(s) to characterize)
Loading weights: 100% 883/883 [00:00<00:00, 4208.99it/s]
Traceback (most recent call last):
  File "/content/rail/image_download/5c_characterize_variants.py", line 299, in <module>
    pipe = pipeline("image-text-to-text", model=MODEL_ID, device=DEVICE, dtype=torch.bfloat16)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/pipelines/__init__.py", line 1084, in pipeline
    processor = _resolve_processor(
                ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/pipelines/__init__.py", line 668, in _resolve_processor
    return _load_pipeline_component(load_processor, processor, load)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python

In [13]:
!python image_download/5c_characterize_variants.py --anchored --fewshot --start 1 --end 5

object address  : 0x7a223c3c0580
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


Compare all variants (plus the zero-shot baseline from above) side by side, over whatever patients each has completed so far — safe to re-run any time, doesn't require any of them to be finished:

In [14]:
import importlib.util
from pathlib import Path

spec = importlib.util.spec_from_file_location("variants", "image_download/5c_characterize_variants.py")
variants_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(variants_mod)

variants_mod.compare_variants({
    "baseline": Path("image_download/nodule_characteristics_gt"),
    "anchored": Path("image_download/nodule_characteristics_gt_anchored"),
    "fewshot": Path("image_download/nodule_characteristics_gt_fewshot"),
    "anchored+fewshot": Path("image_download/nodule_characteristics_gt_anchored_fewshot"),
})

KeyboardInterrupt: 

## Save Results to Google Drive

The files generated in the Colab environment are temporary and will be deleted when the runtime disconnects. To save your results permanently, you can copy them to your mounted Google Drive.

In [15]:
import shutil

source_file = "/content/rail/image_download/nodule_characteristics_gt/LIDC-IDRI-0020.json"
destination_path = "/content/drive/MyDrive/Colab_MedGemma_Results/"

# Create the destination directory if it doesn't exist
import os
os.makedirs(destination_path, exist_ok=True)

# Copy the file
shutil.copy(source_file, destination_path)
print(f"File copied to: {destination_path}")

File copied to: /content/drive/MyDrive/Colab_MedGemma_Results/


## Push to GitHub

To push changes to GitHub, you need to configure your Git identity and provide authentication. It is recommended to use a Personal Access Token (PAT) stored as a Colab secret for security.

First, make sure you have generated a GitHub PAT with `repo` permissions and saved it as a Colab secret named `GH_TOKEN`.

In [48]:
!git lfs uninstall

Hooks for this repository have been removed.
Global Git LFS configuration has been removed.


In [51]:
!git status

On branch medgemma-characterization-variants
Your branch and 'origin/medgemma-characterization-variants' have diverged,
and have 1 and 3 different commits each, respectively.
  (use "git pull" to merge the remote branch into yours)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   image_download/monai_bundles/lung_nodule_ct_detection/models/model.ts

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	image_download/characterize_ground_truth_comparison.csv
	image_download/characterize_ground_truth_summary.json

no changes added to commit (use "git add" and/or "git commit -a")


In [52]:
!printf 'image_download/monai_bundles/lung_nodule_ct_detection/models/* -filter -diff -merge -text\n' >> .git/info/attributes
%env GIT_LFS_SKIP_SMUDGE=1

env: GIT_LFS_SKIP_SMUDGE=1


In [53]:
!git checkout -- image_download/monai_bundles/lung_nodule_ct_detection/models/model.ts
!git status

On branch medgemma-characterization-variants
Your branch and 'origin/medgemma-characterization-variants' have diverged,
and have 1 and 3 different commits each, respectively.
  (use "git pull" to merge the remote branch into yours)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	image_download/characterize_ground_truth_comparison.csv
	image_download/characterize_ground_truth_summary.json

nothing added to commit but untracked files present (use "git add" to track)


In [54]:
!git add image_download/characterize_ground_truth_comparison.csv image_download/characterize_ground_truth_summary.json
!git commit -m "Add characterize_ground_truth summary/comparison"

[medgemma-characterization-variants 4760637] Add characterize_ground_truth summary/comparison
 2 files changed, 112 insertions(+)
 create mode 100644 image_download/characterize_ground_truth_comparison.csv
 create mode 100644 image_download/characterize_ground_truth_summary.json


In [55]:
!git status
!git config pull.rebase true
!git pull

On branch medgemma-characterization-variants
Your branch and 'origin/medgemma-characterization-variants' have diverged,
and have 2 and 3 different commits each, respectively.
  (use "git pull" to merge the remote branch into yours)

nothing to commit, working tree clean
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 842 bytes | 842.00 KiB/s, done.
From https://github.com/freya-gul/rail
   2eb215e..855fe13  medgemma-characterization-variants -> origin/medgemma-characterization-variants
Successfully rebased and updated refs/heads/medgemma-characterization-variants.


In [56]:
!git push origin medgemma-characterization-variants


Enumerating objects: 31, done.
Counting objects: 100% (31/31), done.
Delta compression using up to 12 threads
Compressing objects: 100% (29/29), done.
Writing objects: 100% (29/29), 31.95 KiB | 3.55 MiB/s, done.
Total 29 (delta 21), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (21/21), completed with 2 local objects.
To https://github.com/freya-gul/rail.git
   855fe13..64aef3c  medgemma-characterization-variants -> medgemma-characterization-variants
